## Create Annotation Files 
REGENIE Step 2 requires annotation, set, and mask files for gene-based testing. <br>
<br>
**Annotation file** - functional annotation for each variant given a gene set. Each line contains a variant ID, gene name/gene ID, and a single annotation label <br>
<br>
**Set file** - defines how sets - i.e genes are to be constructed by listing variants corresponding to each gene. Each line contains the gene name, followed by a chromosome, and a position (only representative/for labelling not for defining gene boundaries), then a list of variants included in the gene <br>
<br>
**Mask file** - specifies which annotation label should be combined into masks. Each line contains a mask name, followed by a comma-separated list of annotation included in the mask. Each mask can be defined based on your required analyses criteria

In [ ]:
!pip install pyarrow scikit-learn

In [2]:
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
import matplotlib.pyplot as plt
import numpy as np

Load most severe consequence for each variant

In [3]:
consequences = pd.read_csv('/mnt/project/Annotations/LoFs_most_severe_conseq/ukb23159_b0_v1_sites_annotated_loftee_most_severe_consequences.csv', engine='pyarrow')

In [ ]:
consequences.head()

Load annotations deposited by UKB RAP already

In [5]:
original = pd.read_table('/mnt/project/Bulk/Exome sequences/Population level exome OQFE variants, PLINK format - final release/helper_files/ukb23158_500k_OQFE.annotations.txt.gz', engine='pyarrow', sep=' ', header=None)
original.columns = ['ID', 'Gene_ID', 'Consequence']
original['Gene_Name'] = original.Gene_ID.str.replace(r'\(.*$', '', regex=True)
original.Gene_ID = original.Gene_ID.str.extract(r'(ENSG[0-9]+)')

In [ ]:
original.head()

## Misannotation probabilities 
Annotate each pLoF based on misannotation probabilities calculated by Tony and Jeff's paper (https://pmc.ncbi.nlm.nih.gov/articles/PMC10312940/) <br>
Data deposited at https://zenodo.org/records/7939768

In [ ]:
%%bash

curl -L https://zenodo.org/records/7939768/files/gnomad_lofs_with_misannotation_probabilities.tsv.gz?download=1 -o gnomad_lofs_with_misannotation_probabilities.tsv.gz

In [8]:
misannotation = pd.read_table('gnomad_lofs_with_misannotation_probabilities.tsv.gz', engine='pyarrow')

In [ ]:
misannotation.head()

In [10]:
misannotation_loci = misannotation[['chr', 'start', 'end', 'locus']].drop_duplicates()
misannotation_loci.chr = 'chr' + misannotation_loci.chr.astype(str)

misannotation_loci.to_csv('misannotation_loci_hg19.bed', sep='\t', header=None, index=None)
del misannotation_loci

In [ ]:
!dx upload misannotation_loci_hg19.bed --path /create_annot/

In [ ]:
%%bash 
curl https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz -o hg19ToHg38.over.chain.gz
gunzip hg19ToHg38.over.chain.gz
dx upload hg19ToHg38.over.chain --path /create_annot/

Lift over the hg19 coordinates (gnomad has pLOFs annotated in hg19 ref genome ) to GRCh38 coordinates to work with them <br> UCSC liftover tool was used (docker image code). The file already contains `start` and `end` positions in a 0-indexed format, which is required for `BED` files. <br>
Docker codes for liftOver : `liftOver_docker_download.sh` & `liftover.sh` (https://nyulangone-my.sharepoint.com/:f:/g/personal/srigouri_rajaram_nyulangone_org/EuS6X_SmGI5Ip482eJ0ufXcBkVhdF1PGjsBDiUeu0mBxDA?e=8ttjtr)

In [11]:
misannotation_hg38 = pd.read_table('/mnt/project/create_annot/misannotation_loci_hg38.bed', header=None, engine='pyarrow')
misannotation_hg38.columns = ['chr', 'start0', 'start_hg38', 'locus_hg19']
misannotation_hg38.chr = misannotation_hg38.chr.str.replace(r'^chr', '', regex=True)
misannotation_hg38['locus_hg38'] = misannotation_hg38.chr + ':' + misannotation_hg38.start_hg38.astype(str)

misannotation = pd.merge(misannotation, misannotation_hg38[['start_hg38', 'locus_hg19', 'locus_hg38']], left_on='locus', right_on='locus_hg19')
del misannotation_hg38

misannotation = misannotation[['ensg', 'locus_hg38', 'ref', 'alt', 'p_misannotation']]
misannotation.alt = misannotation.alt.apply(lambda x: eval(x))
misannotation = misannotation[~pd.isna(misannotation.p_misannotation)]

## Constraint Metrics
Each gene is constrained to a different extent by natural selection. These $s_{\mathrm{het}}$ estimates can be accessed from their [Zenodo repository](https://zenodo.org/records/7939768)

In [ ]:
%%bash

curl -L https://zenodo.org/records/7939768/files/s_het_estimates.genebayes.tsv?download=1 -o s_het_estimates.genebayes.tsv

In [13]:
shet = pd.read_table('s_het_estimates.genebayes.tsv', engine='pyarrow')

consequence_genes = consequences[['Gene_ID']].drop_duplicates()
consequence_genes.columns = ['ensg']

constraint = pd.merge(
    shet[['ensg', 'post_mean']],
    consequence_genes,
    on='ensg', how='outer'
)

del consequence_genes

misannotation_genes = misannotation[['ensg']].drop_duplicates()

constraint = pd.merge(
    constraint,
    misannotation_genes,
    on='ensg', how='outer'
)

del misannotation_genes

constraint.columns = ['Gene_ID', 'shet']

del shet

In [ ]:
constraint.head()

Mean imputation for all genes that are missing $s_{\mathrm{het}}$ values. This ends up giving a value of $s_{\mathrm{het}} \approx 0.05$

In [15]:
constraint.loc[pd.isna(constraint.shet), 'shet'] = constraint.shet.mean()

## Variant allele frequencies
Each variant's allele frequency (`wes_allele_frequencies_gr_1.sh` in [create_annot](https://nyulangone-my.sharepoint.com/:f:/r/personal/srigouri_rajaram_nyulangone_org/Documents/ukb_rap/create_annot?csf=1&web=1&e=5ORE2w)), along with the $s_{\mathrm{het}}$, will be used to impute missing misannotation probability values.

In [16]:
af = pd.read_table('/mnt/project/Genotypes/WES/ukb23159_WES_allele_frequencies.afreq', engine='pyarrow', header=None)
af.columns = ['chr', 'variant', 'ref', 'alt', 'af', 'n_alleles']
af = af[af.variant.isin(consequences.ID)]
af['locus'] = af.variant.str.replace(r':[^:]+:[^:]+$', '', regex=True)
af.locus = af.locus.str.replace(r'^23:', 'X:', regex=True)
af.locus = af.locus.str.replace(r'^24:', 'Y:', regex=True)
af = af[['variant', 'locus', 'ref', 'alt', 'af']]
af = af[~pd.isna(af.af)]

In [ ]:
af.head()

## Impute Misannotation Probabilities
Merge the misannotation data with the allele frequency data from the UKB. I will try to match the reference and alternate alleles. This should restrict me to single nucleotide variants. This leaves me with ~328K variant-gene pairs.

In [ ]:
p_misannot = pd.merge(misannotation.explode('alt'), af, left_on=['locus_hg38', 'ref', 'alt'], right_on=['locus', 'ref', 'alt'])
del p_misannot['locus']

len(p_misannot)

In [ ]:
p_misannot.head()

Next, merge into the constraint data. This will give the $s_{\mathrm{het}}$ for each gene.

In [20]:
p_misannot = pd.merge(p_misannot, constraint, left_on='ensg', right_on='Gene_ID')
del p_misannot['Gene_ID']

Train a KNN to predict the probability of misannotation `(y)` using the $s_{\mathrm{het}}$ and allele frequency (`X`) of the gene-variant pair.

In [21]:
y = p_misannot.p_misannotation.values

X = p_misannot[['shet', 'af']].values

Used the `scikit-learn` implementation of KNN regression. Used the Euclidean distance as the metric and set 10 neighbors.

In [22]:
knn_regressor = KNeighborsRegressor(n_neighbors=10, weights='uniform', p=2, metric='minkowski', n_jobs=-1)
knn_regressor = knn_regressor.fit(X, y)

Used the KNN model to predict the probability of misannotation for all variants that were not present in Tony/Jeff's table. Focussed specifically on pLoF variants.

In [23]:
cons_pred = consequences[consequences.Consequence == 'pLoF']
cons_pred = pd.merge(cons_pred, af, left_on='ID', right_on='variant')
cons_pred = pd.merge(cons_pred, constraint, on='Gene_ID')

cons_pred = cons_pred[~(cons_pred.Gene_ID.isin(p_misannot.ensg) & cons_pred.ID.isin(p_misannot.variant))]

`pred_X` used for predictions

In [24]:
pred_X = cons_pred[['shet', 'af']].values

cons_pred['p_misannotation'] = knn_regressor.predict(pred_X)
cons_pred = cons_pred[['Gene_ID', 'variant', 'p_misannotation', 'af']]
cons_pred.columns = ['ensg', 'variant', 'p_misannotation', 'af']

In [ ]:
cons_pred.head()

In [26]:
p_misannot = pd.concat((p_misannot[['ensg', 'variant', 'p_misannotation', 'af']], cons_pred))

In [ ]:
len(p_misannot)

In [ ]:
p_misannot.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12), nrows=2, dpi=100)

ax[0].plot(np.sort(p_misannot.p_misannotation.values), np.linspace(0, 1, len(p_misannot), endpoint=False), label='Probability of Misannotation')
ax[0].plot(np.sort(p_misannot.af.values), np.linspace(0, 1, len(p_misannot), endpoint=False), label='Allele Frequency')
ax[0].legend()
ax[0].set_xlabel('Probability of Misannotation or Allele Frequency')
ax[0].set_ylabel('Proportion of UKB WES Variants')
ax[0].set_xscale('log')

ax[1].plot(np.sort(p_misannot.p_misannotation.values), np.linspace(0, 1, len(p_misannot), endpoint=False), label='Probability of Misannotation')
ax[1].plot(np.sort(p_misannot.af.values), np.linspace(0, 1, len(p_misannot), endpoint=False), label='Allele Frequency')
ax[1].legend()
ax[1].set_xlabel('Probability of Misannotation or Allele Frequency')
ax[1].set_ylabel('Proportion of UKB WES Variants')

plt.savefig('p_misannot_af_eCDFs.pdf')

In [ ]:
print(f'Proportion of Gene-Variant Pairs Included with Misannotation Probability < 0.1: {(p_misannot.p_misannotation < 0.1).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with Misannotation Probability < 0.05: {(p_misannot.p_misannotation < 0.05).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with Misannotation Probability < 0.01: {(p_misannot.p_misannotation < 0.01).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with Misannotation Probability < 0.005: {(p_misannot.p_misannotation < 0.005).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with Misannotation Probability < 0.001: {(p_misannot.p_misannotation < 0.001).sum() / len(p_misannot):.4f}')

In [ ]:
print(f'Proportion of Gene-Variant Pairs Included with AF < 0.01: {(p_misannot.af < 0.01).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with AF < 0.001: {(p_misannot.af < 0.001).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with AF < 0.0001: {(p_misannot.af < 0.0001).sum() / len(p_misannot):.4f}')
print(f'Proportion of Gene-Variant Pairs Included with AF < 0.00001: {(p_misannot.af < 0.00001).sum() / len(p_misannot):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), nrows=1, dpi=100)

ax.plot(np.sort(p_misannot.p_misannotation[p_misannot.af < 0.01].values), np.linspace(0, 1, len(p_misannot[p_misannot.af < 0.01]), endpoint=False), label='AF < 0.01')
ax.plot(np.sort(p_misannot.p_misannotation[p_misannot.af < 0.001].values), np.linspace(0, 1, len(p_misannot[p_misannot.af < 0.001]), endpoint=False), label='AF < 0.001')
ax.plot(np.sort(p_misannot.p_misannotation[p_misannot.af < 0.0001].values), np.linspace(0, 1, len(p_misannot[p_misannot.af < 0.0001]), endpoint=False), label='AF < 0.0001')
ax.plot(np.sort(p_misannot.p_misannotation[p_misannot.af < 0.00001].values), np.linspace(0, 1, len(p_misannot[p_misannot.af < 0.00001]), endpoint=False), label='AF < 0.00001')
ax.legend()
ax.set_xlabel('Probability of Misannotation')
ax.set_ylabel('Proportion of UKB WES Variants')
ax.set_xscale('log')

plt.savefig('p_misannot_cond_af.png', dpi=300)

Retrieve the misannotation probabiltities values for the variant IDs in the original (UKB RAP deposited) file, and then filter to only those with LoF consequences

In [33]:
original_misannot = pd.merge(p_misannot, original, left_on=['ensg', 'variant'], right_on=['Gene_ID', 'ID'])
del original_misannot['ID']
del original_misannot['Gene_ID']
del original_misannot['Gene_Name']
original_misannot = original_misannot[original_misannot.Consequence == 'LoF']

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

ax.plot(np.sort(original_misannot.p_misannotation.values), np.linspace(0, 1, len(original_misannot), endpoint=False), label='Probability of Misannotation')
ax.plot(np.sort(original_misannot.af.values), np.linspace(0, 1, len(original_misannot), endpoint=False), label='Allele Frequency')
ax.legend()
ax.set_xlabel('Probability of Misannotation or Allele Frequency')
ax.set_ylabel('Proportion of UKB WES Variants')
ax.set_xscale('log')
ax.set_title('Probability of Misannotation in Backman et al.')

plt.savefig('p_misannot_af_eCDFs_original.png', dpi=300)

## Annotation File

In [35]:
consequences = pd.merge(consequences, p_misannot, left_on=['Gene_ID', 'ID'], right_on=['ensg', 'variant'], how='left') #matches rows using gene and variant IDs. All rows from consequences are kept; unmatched rows from p_misannot result in NaN values

In [ ]:
consequences.head()

In [ ]:
len(consequences)

In [ ]:
consequences.tail()

In [ ]:
consequences.to_csv("consequences_gr_full.csv", index=False)

In [ ]:
!dx upload consequences_gr_full.csv --path /create_annot/

Six annotations were created: 
1. `pLoFMisAnnot`: Variants that are likely misannotated as pLoFs
2. `pLoFMisAnnot.0.1`: Variants with a probability of misannotation between 0.05 and 0.1
3. `pLoFMisAnnot.0.05`: Variants with a probability of misannotation between 0.01 and 0.05
4. `pLoFMisAnnot.0.01`: Variants with a probability of misannotation less than 0.01
5. `missense`: Missense variants.
6. `synonymous`: Synonymous variants.

In [34]:
consequences['MisAnnot'] = (consequences.Consequence == 'pLoF').map({True: 'MisAnnot', False: 'NotLoF'})
consequences.loc[consequences.p_misannotation < 0.1, 'MisAnnot'] = 'MisAnnot.0.1'
consequences.loc[consequences.p_misannotation < 0.05, 'MisAnnot'] = 'MisAnnot.0.05'
consequences.loc[consequences.p_misannotation < 0.01, 'MisAnnot'] = 'MisAnnot.0.01'

In [40]:
filtered_conseq = consequences.dropna(subset=['ensg', 'variant', 'p_misannotation', 'af'])

In [ ]:
len(filtered_conseq)

In [ ]:
filtered_conseq.head()

In [43]:
filtered_conseq.to_csv("filtered_consequence_gr.csv", index = False) #list of annnotated variants with misannotation probabilities value

In [ ]:
!dx upload ./filtered_consequence_gr.csv --path /create_annot/

In [35]:
consequences['Annotation'] = np.where(consequences.MisAnnot == 'NotLoF', consequences.Consequence, 'pLoF' + consequences.MisAnnot)

In [36]:
consequences[['ID', 'Gene_ID', 'Annotation']].to_csv('lof_annotations.tsv', header=None, index=None, sep='\t')

In [ ]:
df_lof = pd.read_csv("lof_annotations.tsv", sep="\t")

In [ ]:
df_lof.drop_duplicates()

In [ ]:
!dx upload lof_annotations.tsv --path /create_annot/

In [ ]:
df_lof.to_csv("lof_annotations_final.tsv", header=None, index=None, sep="\t")

In [ ]:
!dx upload lof_annotations_final.tsv --path /create_annot/

In [ ]:
!dx upload p_misannot_af_eCDFs_original.png --path /create_annot/

In [ ]:
!dx upload p_misannot_af_eCDFs.pdf --path /create_annot/

In [ ]:
!dx upload p_misannot_cond_af.png --path /create_annot/

In [ ]:
!dx upload s_het_estimates.genebayes.tsv --path /create_annot/

## Set File

In [ ]:
!wget 'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_39/gencode.v39.chr_patch_hapl_scaff.basic.annotation.gtf.gz'

GENCODE GTF file provides the genomic locatios, and types of genes, allowing to filter for protein-coding genes, and get their midpoints

In [46]:
gencode = pd.read_table('gencode.v39.chr_patch_hapl_scaff.basic.annotation.gtf.gz', sep='\t', comment='#', header=None)
gencode.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
gencode = gencode[gencode.feature == 'gene']
gencode.seqname = gencode.seqname.str.replace(r'^chr', '', regex=True)
gencode['midpoint'] = (gencode.start + gencode.end) // 2
gencode['gene_id'] = gencode.attribute.str.extract(r'gene_id "([A-Z]+[0-9]+).[0-9]+";')
gencode['gene_type'] = gencode.attribute.str.extract(r'gene_type "([a-zA-Z_]+)";')

All gene IDs present in the consequences data are also present in the GENCODE data set (which is why `True`)

In [ ]:
consequences.Gene_ID.isin(gencode.gene_id).all()

Restrict to protein-coding genes, which results in 19,408 genes

In [48]:
sets = gencode[gencode.gene_type == 'protein_coding'][['gene_id', 'seqname', 'midpoint']]

sets_vars = consequences.groupby('Gene_ID').apply(lambda x: ','.join(x.ID.values)).reset_index()
sets_vars.columns = ['gene_id', 'variants']

sets = pd.merge(sets, sets_vars, on='gene_id')

del sets_vars

In [ ]:
sets.head()

In [ ]:
len(sets)

In [49]:
sets.to_csv('lof_sets.tsv', header=None, index=None, sep='\t')

In [ ]:
!dx upload lof_sets.tsv --path /create_annot/

## Mask File
The masks are the following.

1. `M1`: All pLoFs
2. `M2`: All pLoFs with misannotation probability less than 0.1
3. `M3`: All pLoFs with misannotation probability less than 0.05
4. `M4`: All pLoFs with misannotation probability less than 0.01
5. `M5`: All synonymous variants

In [51]:
pd.DataFrame({
    'Mask_ID': ['M1', 'M2', 'M3', 'M4', 'M5'],
    'Sets': [
        'pLoFMisAnnot,pLoFMisAnnot.0.1,pLoFMisAnnot.0.05,pLoFMisAnnot.0.01',
        'pLoFMisAnnot.0.1,pLoFMisAnnot.0.05,pLoFMisAnnot.0.01',
        'pLoFMisAnnot.0.05,pLoFMisAnnot.0.01',
        'pLoFMisAnnot.0.01',
        'synonymous'
    ]
}).to_csv('lof_masks.tsv', header=None, index=None, sep='\t')

In [ ]:
!dx upload lof_masks.tsv --path /create_annot/

In [53]:
pd.DataFrame({
    'Mask_ID': ['M1'],
    'Sets': ['LoF']
}).to_csv('lof_masks_original.tsv', header=None, index=None, sep='\t')

In [ ]:
!dx upload lof_masks_original.tsv --path /create_annot/